# Peak Force Heatmaps

Plot peak flat-leg force distributions from the cohesion/friction-angle/omega sweep.

Input:
`../data/synthetic_data/cohesion_phi_omega_force_sweep/force_sweep_summary.csv`

Output:
`../data/synthetic_data/cohesion_phi_omega_force_sweep/force_peak_heatmaps.png`

In [1]:
from pathlib import Path
import csv

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SWEEP_DIR = ROOT / "data" / "synthetic_data" / "cohesion_phi_omega_force_sweep"
SUMMARY_CSV = SWEEP_DIR / "force_sweep_summary.csv"
FIG_PATH = SWEEP_DIR / "force_peak_heatmaps.png"

print(f"Reading: {SUMMARY_CSV}")
print(f"Figure will be saved to: {FIG_PATH}")

Reading: /Users/daweixu/Desktop/Git_Repository/project_MPM_robot_soil/data/synthetic_data/cohesion_phi_omega_force_sweep/force_sweep_summary.csv
Figure will be saved to: /Users/daweixu/Desktop/Git_Repository/project_MPM_robot_soil/data/synthetic_data/cohesion_phi_omega_force_sweep/force_peak_heatmaps.png


In [2]:
def load_summary(path):
    rows = []
    with path.open(newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append({
                "phi_deg": float(row["phi_deg"]),
                "cohesion_Pa": float(row["cohesion_Pa"]),
                "omega_rad_s": float(row["omega_rad_s"]),
                "Fz_peak_N": float(row["Fz_peak_N"]),
                "Fx_peak_N": float(row["Fx_peak_N"]),
                "Fz_peak_angle_deg": float(row["Fz_peak_angle_deg"]),
                "Fz_integral_Ndeg": float(row["Fz_integral_Ndeg"]),
            })
    return rows

rows = load_summary(SUMMARY_CSV)
print(f"Loaded {len(rows)} sweep rows")

phis = sorted({row["phi_deg"] for row in rows})
cohesions = sorted({row["cohesion_Pa"] for row in rows})
omegas = sorted({row["omega_rad_s"] for row in rows})

print("phi values:", phis)
print("cohesion values:", cohesions)
print("omega values:", omegas)

Loaded 72 sweep rows
phi values: [10.0, 15.0, 20.0, 25.0, 30.0, 35.0]
cohesion values: [0.0, 5000.0, 10000.0, 15000.0, 20000.0, 25000.0]
omega values: [10.0, 15.0]


In [3]:
def heatmap_array(rows, omega, value_key):
    arr = np.full((len(cohesions), len(phis)), np.nan, dtype=float)
    phi_to_j = {phi: j for j, phi in enumerate(phis)}
    c_to_i = {coh: i for i, coh in enumerate(cohesions)}

    for row in rows:
        if abs(row["omega_rad_s"] - omega) > 1e-12:
            continue
        i = c_to_i[row["cohesion_Pa"]]
        j = phi_to_j[row["phi_deg"]]
        arr[i, j] = row[value_key]
    return arr


def annotate_heatmap(ax, arr):
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return
    threshold = 0.5 * (float(np.nanmin(finite)) + float(np.nanmax(finite)))
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            val = arr[i, j]
            if not np.isfinite(val):
                continue
            color = "white" if val > threshold else "black"
            ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=7, color=color)


def draw_panel(ax, arr, title, cmap):
    im = ax.imshow(arr, origin="lower", aspect="auto", cmap=cmap)
    ax.set_title(title)
    ax.set_xticks(np.arange(len(phis)))
    ax.set_xticklabels([f"{phi:g}" for phi in phis])
    ax.set_yticks(np.arange(len(cohesions)))
    ax.set_yticklabels([f"{coh:g}" for coh in cohesions])
    ax.set_xlabel("Friction angle phi (deg)")
    ax.set_ylabel("Cohesion c (Pa)")
    annotate_heatmap(ax, arr)
    return im

In [4]:
fig, axes = plt.subplots(2, len(omegas), figsize=(6 * len(omegas), 9), squeeze=False)
fig.suptitle("Peak Raw Force Heatmaps for Flat-Leg Sweep", fontsize=14, y=1.02)

for col, omega in enumerate(omegas):
    fz = heatmap_array(rows, omega, "Fz_peak_N")
    fx = heatmap_array(rows, omega, "Fx_peak_N")

    im_fz = draw_panel(axes[0, col], fz, f"Peak Fz (N), omega={omega:g} rad/s", "viridis")
    im_fx = draw_panel(axes[1, col], fx, f"Peak Fx (N), omega={omega:g} rad/s", "magma")

    fig.colorbar(im_fz, ax=axes[0, col], fraction=0.046, pad=0.04, label="N")
    fig.colorbar(im_fx, ax=axes[1, col], fraction=0.046, pad=0.04, label="N")

plt.tight_layout()
plt.show()
plt.savefig(FIG_PATH, dpi=180, bbox_inches="tight")

print(f"Saved: {FIG_PATH}")

/var/folders/jq/khx6mlw900l3dzqh_x5_9v540000gn/T/ipykernel_33644/3144140161.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: /Users/daweixu/Desktop/Git_Repository/project_MPM_robot_soil/data/synthetic_data/cohesion_phi_omega_force_sweep/force_peak_heatmaps.png


In [5]:
# Optional: print the strongest cases for a quick numeric check.
for omega in omegas:
    subset = [row for row in rows if abs(row["omega_rad_s"] - omega) < 1e-12]
    best_fz = max(subset, key=lambda row: row["Fz_peak_N"])
    best_fx = max(subset, key=lambda row: row["Fx_peak_N"])
    print(f"omega={omega:g} rad/s")
    print(
        f"  max Fz: {best_fz['Fz_peak_N']:.2f} N "
        f"at c={best_fz['cohesion_Pa']:.0f} Pa, phi={best_fz['phi_deg']:.0f} deg"
    )
    print(
        f"  max Fx: {best_fx['Fx_peak_N']:.2f} N "
        f"at c={best_fx['cohesion_Pa']:.0f} Pa, phi={best_fx['phi_deg']:.0f} deg"
    )

omega=10 rad/s
  max Fz: 130.34 N at c=25000 Pa, phi=30 deg
  max Fx: 229.97 N at c=25000 Pa, phi=10 deg
omega=15 rad/s
  max Fz: 166.53 N at c=25000 Pa, phi=35 deg
  max Fx: 285.34 N at c=20000 Pa, phi=10 deg
